# PETase 3D Structure Acquisition

**Strategy (per sequence):**
1. Search in RCSB PDB via Sequence Search API (BLAST-like). If there is a match with identity >= cutoff (re-verified from the downloaded .pdb file, not just relying on the API field) -> use the experimental PDB structure.
2. If there is no sufficiently identical match -> predict with ESMFold (facebook/esmfold_v1).

**Output (automatic, no manual merging required):**
- `structures/pdb_experimental/*.pdb` + `structures/esmfold_predicted/*.pdb`
- `structures/structure_metadata_merged.csv` -> master file, all sequences, `structure_source` column indicates the origin of the structure (`PDB_experimental` / `ESMFold_predicted` / `None` if both failed)
- `structures/structure_metadata_pdbexp.csv` -> subset of PDB_experimental rows only (derived from merged)
- `structures/structure_metadata_esmfold.csv` -> subset of ESMFold_predicted rows only (derived from merged)

Supports **resume**: if the Colab runtime disconnects, just re-run the acquisition cell, sequences that already have a structure (from the `structure_metadata_merged.csv` checkpoint) are automatically skipped.

In [ ]:
#@title # Mount Google Drive & Install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q biopython requests transformers accelerate einops tqdm pandas numpy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.2 MB/s eta 0:00:00


In [ ]:
#@title # 1. Configuration

import os
import time
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- Adjust project path here if different ---
PROJECT_ROOT = "/content/drive/MyDrive/InterPET"
DATASETS_DIR = os.path.join(PROJECT_ROOT, "datasets")
STRUCT_DIR = os.path.join(PROJECT_ROOT, "structures")
PDB_DIR = os.path.join(STRUCT_DIR, "pdb_experimental")
ESMFOLD_DIR = os.path.join(STRUCT_DIR, "esmfold_predicted")
os.makedirs(PDB_DIR, exist_ok=True)
os.makedirs(ESMFOLD_DIR, exist_ok=True)

# Input: dataset curation results (from Dataset Curation & QC notebook)
TRAIN_FINAL_CSV = os.path.join(DATASETS_DIR, "train_final_labels.csv")
BENCHMARK_FINAL_CSV = os.path.join(DATASETS_DIR, "benchmark_final_labels.csv")

# Output -- one master file + 2 derivatives, automatic, no manual merge needed
METADATA_MERGED = os.path.join(STRUCT_DIR, "structure_metadata_merged.csv")
METADATA_PDBEXP = os.path.join(STRUCT_DIR, "structure_metadata_pdbexp.csv")
METADATA_ESMFOLD = os.path.join(STRUCT_DIR, "structure_metadata_esmfold.csv")

# -----------------------------------------------------------------------------
# PDB search parameters
# -----------------------------------------------------------------------------
PDB_IDENTITY_CUTOFF = 0.95   # min. identity to query sequence to be considered a "match"
PDB_EVALUE_CUTOFF = 1.0      # RCSB sequence search e-value cutoff (loose, further filtered by identity)
MAX_STRUCT_LEN = 500         # residues; longer sequences are truncated for ESMFold
MAX_HITS_TO_TRY = 3          # try several top candidates if candidate #1 fails verification/download
REQUEST_SLEEP = 0.3          # delay between requests to RCSB API (polite to public servers)

print("Config OK.")
print(f"PDB experimental -> {PDB_DIR}")
print(f"ESMFold predicted -> {ESMFOLD_DIR}")
print(f"Merged metadata   -> {METADATA_MERGED}")
print(f"PDB-only metadata -> {METADATA_PDBEXP}")
print(f"ESMFold metadata  -> {METADATA_ESMFOLD}")

Config OK.
PDB experimental -> /content/drive/MyDrive/InterPET/structures/pdb_experimental
ESMFold predicted -> /content/drive/MyDrive/InterPET/structures/esmfold_predicted
Merged metadata   -> /content/drive/MyDrive/InterPET/structures/structure_metadata_merged.csv
PDB-only metadata -> /content/drive/MyDrive/InterPET/structures/structure_metadata_pdbexp.csv
ESMFold metadata  -> /content/drive/MyDrive/InterPET/structures/structure_metadata_esmfold.csv


In [ ]:
#@title # 2. Load sequences from curated dataset

dfs = []
if os.path.exists(TRAIN_FINAL_CSV):
    df_train = pd.read_csv(TRAIN_FINAL_CSV)
    df_train["split"] = "train"
    dfs.append(df_train[["id", "sequence", "label", "split"]])
    print(f"Train: {len(df_train)} sequences")
else:
    print(f"[WARN] {TRAIN_FINAL_CSV} not found -- skipped")

if os.path.exists(BENCHMARK_FINAL_CSV):
    df_bench = pd.read_csv(BENCHMARK_FINAL_CSV)
    df_bench["split"] = "benchmark"
    dfs.append(df_bench[["id", "sequence", "label", "split"]])
    print(f"Benchmark: {len(df_bench)} sequences")
else:
    print(f"[WARN] {BENCHMARK_FINAL_CSV} not found -- skipped")

if not dfs:
    raise FileNotFoundError("Dataset not found. Please run the 'Dataset Curation & QC' notebook first.")

df_seqs = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="sequence").reset_index(drop=True)
print(f"\nTotal unique sequences (train+benchmark): {len(df_seqs)}")

Train: 937 sequences
Benchmark: 139 sequences

Total unique sequences (train+benchmark): 1076


In [ ]:
#@title # 3. RCSB PDB Sequence Search API + downloader (FIXED)
# Bug fix notes (from previous notebook, maintained here):
# - The correct query parameter for RCSB Search API v2 is "sequence_type" (not "target").
# - RCSB filters results on the SERVER side based on the identity_cutoff & evalue_cutoff we send
#   -- so all entries in 'result_set' are ALREADY >= identity_cutoff.
# - The per-hit field in 'result_set' only contains {'identifier': 'PDBID_ENTITYID', 'score': float}.
#   Actual identity is recalculated from the downloaded sequence (see compute_identity below),
#   not blindly trusted from the API response.

RCSB_SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_ENTRY_URL = "https://data.rcsb.org/rest/v1/core/entry/{}"
RCSB_DOWNLOAD_URL = "https://files.rcsb.org/download/{}.pdb"


def search_pdb_by_sequence(sequence, identity_cutoff=PDB_IDENTITY_CUTOFF, evalue_cutoff=PDB_EVALUE_CUTOFF):
    """Search for PDB structures identical/highly similar to the query sequence.
    Return list of dict {pdb_id, entity_id, score}, already sorted by score desc.
    """
    query = {
        "query": {
            "type": "terminal",
            "service": "sequence",
            "parameters": {
                "evalue_cutoff": evalue_cutoff,
                "identity_cutoff": identity_cutoff,
                "sequence_type": "protein",
                "value": sequence,
            },
        },
        "return_type": "polymer_entity",
        "request_options": {
            "results_content_type": ["experimental"],
            "sort": [{"sort_by": "score", "direction": "desc"}],
        },
    }
    try:
        resp = requests.post(RCSB_SEARCH_URL, json=query, timeout=30)
        if resp.status_code == 204:  # no hits
            return []
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"    [WARN] RCSB search error: {e}")
        return []

    hits = []
    for r in data.get("result_set", []):
        identifier = r["identifier"]  # format: "PDBID_ENTITYID"
        pdb_id, entity_id = identifier.split("_")
        hits.append({"pdb_id": pdb_id, "entity_id": entity_id, "score": r.get("score")})

    return hits


def get_entry_resolution(pdb_id):
    """Retrieve the resolution (Angstrom) of a PDB entry, if available (X-ray only)."""
    try:
        resp = requests.get(RCSB_ENTRY_URL.format(pdb_id), timeout=15)
        resp.raise_for_status()
        data = resp.json()
        res = data.get("rcsb_entry_info", {}).get("resolution_combined")
        return res[0] if res else None
    except Exception:
        return None


def download_pdb(pdb_id, out_path):
    """Download .pdb file from RCSB. Return True if successful."""
    try:
        resp = requests.get(RCSB_DOWNLOAD_URL.format(pdb_id), timeout=30)
        resp.raise_for_status()
        with open(out_path, "w") as f:
            f.write(resp.text)
        return True
    except Exception as e:
        print(f"    [WARN] Failed to download {pdb_id}: {e}")
        return False


print("RCSB PDB search/download functions ready (FIXED).")

RCSB PDB search/download functions ready (FIXED).


In [ ]:
#@title # 4. Verify actual identity from downloaded PDB structures
# We verify the ACTUAL identity by aligning the query sequence with the sequence
# extracted from the downloaded .pdb file. This acts as a safety-net if a high-scoring hit
# turns out to have low actual identity.

from Bio.PDB import PDBParser, PPBuilder
from Bio.Align import PairwiseAligner
import warnings
from Bio import BiopythonWarning

warnings.simplefilter("ignore", BiopythonWarning)

_aligner = PairwiseAligner()
_aligner.mode = "global"
_aligner.match_score = 2
_aligner.mismatch_score = -1
_aligner.open_gap_score = -10
_aligner.extend_gap_score = -0.5

_pdb_parser = PDBParser(QUIET=True)
_ppb = PPBuilder()


def get_chain_sequences(pdb_path):
    """Extract protein sequences for each chain from the downloaded .pdb file."""
    structure = _pdb_parser.get_structure("s", pdb_path)
    seqs = []
    for pp in _ppb.build_peptides(structure):
        seqs.append(str(pp.get_sequence()))
    return seqs


def compute_identity(query_seq, target_seq):
    """Calculate % identity (0-1) via simple global alignment."""
    if not target_seq:
        return 0.0
    aln = _aligner.align(query_seq, target_seq)[0]
    aligned_a, aligned_b = aln[0], aln[1]
    matches = sum(1 for a, b in zip(aligned_a, aligned_b) if a == b and a != "-")
    denom = min(len(query_seq), len(target_seq))
    return matches / denom if denom else 0.0


def best_identity_in_pdb(query_seq, pdb_path):
    """Highest identity between query_seq and all chains in pdb_path."""
    best = 0.0
    for chain_seq in get_chain_sequences(pdb_path):
        ident = compute_identity(query_seq, chain_seq)
        if ident > best:
            best = ident
    return best


print("Identity verification functions ready.")

Identity verification functions ready.


In [ ]:
#@title # 5. ESMFold fallback (for sequences without PDB match)

import torch
from transformers import AutoTokenizer, EsmForProteinFolding

ESMFOLD_MODEL_NAME = "facebook/esmfold_v1"
ESMFOLD_CHUNK_SIZE = 64
device = "cuda" if torch.cuda.is_available() else "cpu"

_esmfold_tokenizer = None
_esmfold_model = None


def _load_esmfold():
    global _esmfold_tokenizer, _esmfold_model
    if _esmfold_model is not None:
        return
    print("  Loading ESMFold model (several GB, only once)...")
    _esmfold_tokenizer = AutoTokenizer.from_pretrained(ESMFOLD_MODEL_NAME)
    _esmfold_model = EsmForProteinFolding.from_pretrained(ESMFOLD_MODEL_NAME, low_cpu_mem_usage=True)
    _esmfold_model = _esmfold_model.to(device)
    _esmfold_model.esm = _esmfold_model.esm.half()
    _esmfold_model.trunk.set_chunk_size(ESMFOLD_CHUNK_SIZE)
    _esmfold_model.eval()


@torch.no_grad()
def predict_structure_esmfold(sequence, out_path):
    """Predict structure with ESMFold, save as PDB file.
    Return mean pLDDT (0-100) if successful, None if failed.
    """
    _load_esmfold()
    seq = sequence[:MAX_STRUCT_LEN]
    try:
        pdb_str = _esmfold_model.infer_pdb(seq)

        bfactors = []
        for line in pdb_str.splitlines():
            if line.startswith("ATOM"):
                try:
                    bfactors.append(float(line[60:66]))
                except ValueError:
                    pass
        mean_plddt = float(np.mean(bfactors)) if bfactors else None

        # Fix: some versions of transformers write B-factors as fractions 0-1, not standard pLDDT scale 0-100.
        # Original pLDDT cannot always be <= 1.5, so if detected, normalize to 0-100 before saving.
        if mean_plddt is not None and mean_plddt <= 1.5:
            bfactors = [b * 100 for b in bfactors]
            mean_plddt = float(np.mean(bfactors))
            pdb_str = "\n".join(
                line[:60] + f"{float(line[60:66]) * 100:6.2f}" + line[66:]
                if line.startswith("ATOM") else line
                for line in pdb_str.splitlines()
            )

        with open(out_path, "w") as f:
            f.write(pdb_str)

        return mean_plddt
    except Exception as e:
        print(f"    [WARN] ESMFold failed for sequence ({e})")
        return None


print("ESMFold fallback function ready (model will be loaded on first use).")

ESMFold fallback function ready (model will be loaded on first use).


KeyboardInterrupt: 

In [ ]:
#@title # 6. Run structure acquisition for the entire dataset (PDB -> ESMFold fallback)

metadata_rows = []

# Resume support: if METADATA_MERGED already exists, skip already processed sequences
already_done = set()
if os.path.exists(METADATA_MERGED):
    df_existing = pd.read_csv(METADATA_MERGED)
    # Ensure df_existing only has one entry per 'id', prioritizing PDB_experimental if available
    # Sort by 'structure_source' to put 'PDB_experimental' first, then drop duplicates
    df_existing_clean = df_existing.sort_values(
        by=['structure_source'],
        key=lambda x: x.map({'PDB_experimental': 0, 'ESMFold_predicted': 1, None: 2}).fillna(2)
    ).drop_duplicates(subset=['id'], keep='first')

    already_done = set(df_existing_clean["id"])
    metadata_rows = df_existing_clean.to_dict("records")
    print(f"Resuming from checkpoint: {len(already_done)} sequences already processed.")

n_pdb_found = 0
n_esmfold_used = 0
n_failed = 0

for _, row in tqdm(df_seqs.iterrows(), total=len(df_seqs), desc="Structure acquisition"):
    seq_id = row["id"]
    sequence = row["sequence"]

    if seq_id in already_done:
        continue

    safe_id = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(seq_id))

    record = {
        "id": seq_id, "sequence": sequence, "split": row["split"], "label": row["label"],
        "structure_source": None, "pdb_id": None, "identity_pct": None,
        "resolution": None, "plddt_mean": None, "file_path": None,
    }

    # --- 1. Try searching in PDB first ---
    hits = search_pdb_by_sequence(sequence)
    time.sleep(REQUEST_SLEEP)

    for hit in hits[:MAX_HITS_TO_TRY]:
        pdb_id = hit["pdb_id"]
        tmp_path = os.path.join(PDB_DIR, f"{safe_id}_{pdb_id}.pdb")
        ok = download_pdb(pdb_id, tmp_path)
        time.sleep(REQUEST_SLEEP)
        if not ok:
            continue

        real_identity = best_identity_in_pdb(sequence, tmp_path)
        if real_identity >= PDB_IDENTITY_CUTOFF:
            resolution = get_entry_resolution(pdb_id)
            time.sleep(REQUEST_SLEEP)
            record.update({
                "structure_source": "PDB_experimental",
                "pdb_id": pdb_id,
                "identity_pct": round(real_identity * 100, 2),
                "resolution": resolution,
                "file_path": tmp_path,
            })
            n_pdb_found += 1
            break
        else:
            os.remove(tmp_path)  # not a real match after verification, delete the file

    # --- 2. Fallback to ESMFold if no sufficiently identical PDB match ---
    if record["structure_source"] is None:
        out_path = os.path.join(ESMFOLD_DIR, f"{safe_id}.pdb")
        plddt = predict_structure_esmfold(sequence, out_path)
        if plddt is not None:
            record.update({
                "structure_source": "ESMFold_predicted",
                "plddt_mean": plddt,
                "file_path": out_path,
            })
            n_esmfold_used += 1
        else:
            n_failed += 1

    metadata_rows.append(record)

    # Checkpoint every 25 sequences (just in case runtime disconnects)
    if len(metadata_rows) % 25 == 0:
        pd.DataFrame(metadata_rows).to_csv(METADATA_MERGED, index=False)

pd.DataFrame(metadata_rows).to_csv(METADATA_MERGED, index=False)

print("\n" + "=" * 60)
print("STRUCTURE ACQUISITION SUMMARY")
print("=" * 60)
print(f"Total sequences: {len(df_seqs)}")
print(f"  Found in PDB (experimental) : {n_pdb_found}")
print(f"  Predicted via ESMFold         : {n_esmfold_used}")
print(f"  Failed (both)                 : {n_failed}")
print(f"\nMaster metadata saved: {METADATA_MERGED}")

Structure acquisition:   0%|          | 2/1076 [00:29<4:34:00, 15.31s/it]

  Loading ESMFold model (several GB, only once)...


config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

...

    [WARN] Failed to download 8CA9: Response ended prematurely


Structure acquisition:  94%|█████████▍| 1013/1076 [2:41:45<09:48,  9.33s/it]

    [WARN] RCSB search error: HTTPSConnectionPool(host='search.rcsb.org', port=443): Read timed out. (read timeout=30)


Structure acquisition: 100%|██████████| 1076/1076 [2:50:52<00:00,  9.53s/it]


STRUCTURE ACQUISITION SUMMARY
Total sequences: 1076
  Found in PDB (experimental) : 218
  Predicted via ESMFold         : 858
  Failed (both)                 : 0

Master metadata saved: /content/drive/MyDrive/InterPET/structures/structure_metadata_merged.csv


In [ ]:
#@title # 6b Recovery: re-search & download missing PDB experimental structures

import time

def download_pdb_retry(pdb_id, out_path, max_retries=3, backoff=2.0):
    """Similar to download_pdb, but with retry + exponential backoff to handle temporary network/timeout errors to RCSB."""
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(RCSB_DOWNLOAD_URL.format(pdb_id), timeout=30)
            resp.raise_for_status()
            with open(out_path, "w") as f:
                f.write(resp.text)
            return True
        except Exception as e:
            if attempt < max_retries:
                time.sleep(backoff * attempt)
            else:
                print(f"    [WARN] Failed to download {pdb_id} after {max_retries}x attempts: {e}")
    return False


def search_pdb_by_sequence_retry(sequence, identity_cutoff=PDB_IDENTITY_CUTOFF,
                                   evalue_cutoff=PDB_EVALUE_CUTOFF, max_retries=3, backoff=2.0):
    """Similar to search_pdb_by_sequence, but retries if the request fails/timeouts (not if there are no hits -- that's a valid result, not an error)."""
    for attempt in range(1, max_retries + 1):
        try:
            hits = search_pdb_by_sequence(sequence, identity_cutoff, evalue_cutoff)
            return hits
        except Exception as e:
            if attempt < max_retries:
                print(f"    [WARN] Search error, retry {attempt}/{max_retries}: {e}")
                time.sleep(backoff * attempt)
            else:
                print(f"    [WARN] Search still failed after {max_retries}x attempts: {e}")
                return []
    return []


# --- 1. Load existing master metadata ---
assert os.path.exists(METADATA_MERGED), "Run the acquisition cell (part 6) first before recovery."
df_meta = pd.read_csv(METADATA_MERGED)

# --- 2. Determine which sequences need to be re-attempted ---
def _file_missing(p):
    return not (isinstance(p, str) and os.path.exists(p))

# Case A: metadata states PDB_experimental, but the file itself is missing/corrupt on disk
mask_pdb_but_missing = (df_meta["structure_source"] == "PDB_experimental") & (
    df_meta["file_path"].apply(_file_missing)
)
# Case B: sequence does not have PDB_experimental at all (ESMFold fallback / total failure) --
# these are "missed PDB" candidates most relevant to the 221 vs 218 gap
mask_not_pdb = df_meta["structure_source"] != "PDB_experimental"

retry_ids = set(df_meta.loc[mask_pdb_but_missing, "id"]) | set(df_meta.loc[mask_not_pdb, "id"])

print(f"Rows with 'PDB_experimental' but missing files on disk           : {mask_pdb_but_missing.sum()}")
print(f"Rows ESMFold_predicted / total failure (PDB retry candidates)  : {mask_not_pdb.sum()}")
print(f"Total sequences to re-attempt                     : {len(retry_ids)}")

df_retry = df_seqs[df_seqs["id"].isin(retry_ids)].reset_index(drop=True)

# --- 3. Re-run search -> download -> identity verification specifically for these candidates ---
n_recovered = 0

for _, row in tqdm(df_retry.iterrows(), total=len(df_retry), desc="Recovery PDB experimental"):
    seq_id = row["id"]
    sequence = row["sequence"]
    safe_id = "".join(c if c.isalnum() or c in "-_" else "_" for c in str(seq_id))

    hits = search_pdb_by_sequence_retry(sequence)
    time.sleep(REQUEST_SLEEP)

    found = False
    for hit in hits[:MAX_HITS_TO_TRY]:
        pdb_id = hit["pdb_id"]
        tmp_path = os.path.join(PDB_DIR, f"{safe_id}_{pdb_id}.pdb")
        ok = download_pdb_retry(pdb_id, tmp_path)
        time.sleep(REQUEST_SLEEP)
        if not ok:
            continue

        real_identity = best_identity_in_pdb(sequence, tmp_path)
        if real_identity >= PDB_IDENTITY_CUTOFF:
            resolution = get_entry_resolution(pdb_id)
            time.sleep(REQUEST_SLEEP)

            # If this sequence previously had an ESMFold prediction, delete its old file
            # to avoid accumulating unused files
            old_row = df_meta.loc[df_meta["id"] == seq_id]
            if len(old_row) and old_row.iloc[0]["structure_source"] == "ESMFold_predicted":
                old_path = old_row.iloc[0]["file_path"]
                if isinstance(old_path, str) and os.path.exists(old_path):
                    os.remove(old_path)

            new_record = {
                "id": seq_id, "sequence": sequence, "split": row["split"], "label": row["label"],
                "structure_source": "PDB_experimental", "pdb_id": pdb_id,
                "identity_pct": round(real_identity * 100, 2),
                "resolution": resolution, "plddt_mean": None, "file_path": tmp_path,
            }
            df_meta = df_meta[df_meta["id"] != seq_id]  # discard old row for this id (if any)
            df_meta = pd.concat([df_meta, pd.DataFrame([new_record])], ignore_index=True)

            n_recovered += 1
            found = True
            break
        else:
            os.remove(tmp_path)  # not a valid match after re-verification, delete

print(f"\nSuccessfully recovered {n_recovered} sequences as PDB_experimental.")

# --- 4. Save the updated master metadata ---
df_meta.to_csv(METADATA_MERGED, index=False)
print(f"Master metadata updated and saved: {METADATA_MERGED}")
print("\n>>> Rerun cells '7. Create derivative files' and '8. Summary & sanity check' "
      "below so that structure_metadata_pdbexp.csv / structure_metadata_esmfold.csv are also updated.")

Rows with 'PDB_experimental' but missing files on disk           : 0
Rows ESMFold_predicted / total failure (PDB retry candidates)  : 858
Total sequences to re-attempt                     : 858


Recovery PDB experimental:  21%|██        | 158/745 [02:57<11:56,  1.22s/it]/tmp/ipykernel_1457/500367995.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_meta = pd.concat([df_meta, pd.DataFrame([new_record])], ignore_index=True)
Recovery PDB experimental:  96%|█████████▌| 715/745 [16:05<00:53,  1.79s/it]/tmp/ipykernel_1457/500367995.py:103: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_meta = pd.concat([df_meta, pd.DataFrame([new_record])], ignore_index=True)
Recovery PDB experimenta


Successfully recovered 3 sequences as PDB_experimental.
Master metadata updated and saved: /content/drive/MyDrive/InterPET/structures/structure_metadata_merged.csv

>>> Rerun cells '7. Create derivative files' and '8. Summary & sanity check' below so that structure_metadata_pdbexp.csv / structure_metadata_esmfold.csv are also updated.


In [ ]:
#@title # 7. Create derivative files (pdbexp & esmfold) from master -- automatic, no manual merge

df_meta = pd.read_csv(METADATA_MERGED)

df_meta[df_meta["structure_source"] == "PDB_experimental"].to_csv(METADATA_PDBEXP, index=False)
df_meta[df_meta["structure_source"] == "ESMFold_predicted"].to_csv(METADATA_ESMFOLD, index=False)

print(f"Written: {METADATA_PDBEXP}  ({(df_meta['structure_source']=='PDB_experimental').sum()} rows)")
print(f"Written: {METADATA_ESMFOLD} ({(df_meta['structure_source']=='ESMFold_predicted').sum()} rows)")

Written: /content/drive/MyDrive/InterPET/structures/structure_metadata_pdbexp.csv  (221 rows)
Written: /content/drive/MyDrive/InterPET/structures/structure_metadata_esmfold.csv (855 rows)


In [ ]:
#@title # 8. Summary & sanity check

df_meta = pd.read_csv(METADATA_MERGED)
esmfold_rows = df_meta[df_meta["structure_source"] == "ESMFold_predicted"]

for _, row in esmfold_rows.iterrows():
    path = row["file_path"]
    if not (isinstance(path, str) and os.path.exists(path)):
        continue

    with open(path) as f:
        lines = f.readlines()

    raw_bfactors = [float(l[60:66]) for l in lines if l.startswith(("ATOM", "HETATM"))]
    if not raw_bfactors:
        continue
    current_max = max(raw_bfactors)

    if current_max > 100:      # double scaling -> divide by 100
        factor = 1 / 100
    elif current_max <= 1.5:   # not yet scaled -> multiply by 100
        factor = 100
    else:                      # already correct (0-100) -> do nothing
        factor = None

    if factor is not None:
        new_bfactors = []
        for i, line in enumerate(lines):
            if line.startswith(("ATOM", "HETATM")):
                b = float(line[60:66]) * factor
                b = max(0.0, min(999.99, b))
                lines[i] = line[:60] + f"{b:6.2f}" + line[66:]
                new_bfactors.append(b)
        with open(path, "w") as f:
            f.writelines(lines)
        df_meta.loc[df_meta["id"] == row["id"], "plddt_mean"] = np.mean(new_bfactors)

df_meta.to_csv(METADATA_MERGED, index=False)

print("Structure source distribution:")
print(df_meta["structure_source"].value_counts(dropna=False))

print("\nFor experimental PDB structures -- identity% & resolution distribution:")
pdb_rows = df_meta[df_meta["structure_source"] == "PDB_experimental"]
if len(pdb_rows) > 0:
    print(pdb_rows[["identity_pct", "resolution"]].describe())
else:
    print("  (none)")

print("\nFor ESMFold structures -- pLDDT distribution:")
esmfold_rows = df_meta[df_meta["structure_source"] == "ESMFold_predicted"]
if len(esmfold_rows) > 0:
    print(esmfold_rows["plddt_mean"].describe())
    low_conf = (esmfold_rows["plddt_mean"] < 70).sum()
    print(f"\n  {low_conf}/{len(esmfold_rows)} ESMFold predictions have pLDDT < 70 (low confidence)")
    print("  -> consider excluding or treating these sequences carefully during")
    print("     structural feature extraction (predictions may not be reliable).")
else:
    print("  (none)")

n_missing = df_meta["structure_source"].isna().sum()
if n_missing > 0:
    print(f"\n[WARNING] {n_missing} sequences FAILED totally (PDB not found & ESMFold error).")
    print("  Check the empty 'file_path' column in structure_metadata_merged.csv for details.")

Structure source distribution:
structure_source
ESMFold_predicted    855
PDB_experimental     221
Name: count, dtype: int64

For experimental PDB structures -- identity% & resolution distribution:
       identity_pct  resolution
count    221.000000  221.000000
mean      99.232308    1.702181
std        1.102672    0.403459
min       95.000000    0.780000
25%       98.830000    1.400000
50%       99.640000    1.598000
75%      100.000000    1.900000
max      100.000000    3.174000

For ESMFold structures -- pLDDT distribution:
count    855.000000
mean      86.329298
std        8.601604
min       24.831510
25%       84.919836
50%       88.158284
75%       91.212692
max       96.744822
Name: plddt_mean, dtype: float64

  40/855 ESMFold predictions have pLDDT < 70 (low confidence)
  -> consider excluding or treating these sequences carefully during
     structural feature extraction (predictions may not be reliable).
